# Notebook 3 — Indic Parler-TTS

- Repo: https://github.com/huggingface/parler-tts
- Model: https://huggingface.co/ai4bharat/indic-parler-tts
- Description-prompted; **two-tokenizer pattern** (description ↔ Flan-T5 tokenizer; transcript ↔ model prompt tokenizer).
- The DESCRIPTION string is **held constant** across all 30 sentences — varying it per sentence would confound the comparison.
- Strongest Hinglish prior of the five (joint Indic + English 1806 h corpus).


In [ ]:
# === Cell 1: install ===
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q "transformers>=4.46,<4.50" accelerate soundfile sentencepiece


In [ ]:
# Mount Drive (or skip if running locally) and clone the audit folder.
# Adjust this cell to point AUDIT_DIR at wherever audit/ lives in your runtime.
import os
from pathlib import Path

# Two common patterns:
#   1. Colab + Drive: AUDIT_DIR = "/content/drive/MyDrive/hienglish/audit"
#   2. Colab + git clone:
#         !git clone https://github.com/<you>/hienglish.git /content/hienglish
#         AUDIT_DIR = "/content/hienglish/audit"
#   3. Local: AUDIT_DIR = str(Path.cwd().parent / "audit")  (if launched from notebooks/)

AUDIT_DIR = os.environ.get("AUDIT_DIR", "/content/audit")
assert Path(AUDIT_DIR).is_dir(), f"AUDIT_DIR={AUDIT_DIR} missing — set it before running."
print(f"AUDIT_DIR = {AUDIT_DIR}")


In [ ]:
import csv
from pathlib import Path

EVAL_TSV = Path(AUDIT_DIR) / "eval_sentences.tsv"
with open(EVAL_TSV, encoding="utf-8") as f:
    rows = list(csv.DictReader(f, delimiter="\t"))

assert len(rows) == 30, f"expected 30 sentences, got {len(rows)}"
print(f"Loaded {len(rows)} sentences from {EVAL_TSV}")
print(rows[0])


In [ ]:
# === Cell 4: load ===
from pathlib import Path
import time, json
import torch
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_NAME = "indic_parler"
OUT = Path(AUDIT_DIR) / "results" / MODEL_NAME
OUT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32

REPO = "ai4bharat/indic-parler-tts"
model     = ParlerTTSForConditionalGeneration.from_pretrained(REPO, torch_dtype=dtype).to(device)
tok       = AutoTokenizer.from_pretrained(REPO)
desc_tok  = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)
SR = int(model.config.sampling_rate)

# Held constant for fair comparison across all 30 sentences:
DESCRIPTION = (
    "A female speaker delivers a clear, moderately-paced Hindi speech "
    "with neutral expression. The recording is high quality with no background noise."
)
desc_input_ids = desc_tok(DESCRIPTION, return_tensors="pt").input_ids.to(device)
print(f"sampling_rate={SR}, dtype={dtype}, device={device}")


In [ ]:
# === Cell 5: run inference ===
log = []
for r in rows:
    rid, cat, text = r["id"], r["category"], r["text"]
    t0 = time.time()
    try:
        prompt_input_ids = tok(text, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(input_ids=desc_input_ids, prompt_input_ids=prompt_input_ids)
        audio = gen.cpu().to(torch.float32).numpy().squeeze()
        out_path = OUT / f"{rid}.wav"
        sf.write(out_path, audio, SR)
        log.append({
            "id": rid, "category": cat, "sr": SR,
            "duration_s": float(len(audio) / SR),
            "elapsed_s": time.time() - t0,
            "status": "ok",
        })
    except Exception as e:
        log.append({"id": rid, "category": cat, "status": "error", "error": str(e)})
        print(f"  [error] {rid}: {e}")
        # If OOM on long sentences: clear cache and continue
        torch.cuda.empty_cache()


In [ ]:
import json
out_log = Path(OUT) / "log.json"
with open(out_log, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

n_ok = sum(1 for x in log if x["status"] == "ok")
print(f"{MODEL_NAME}: {n_ok}/30 succeeded — log at {out_log}")
